In [1]:
# R-1 — Environment. Afterwards: Runtime → Restart session, then run R-2 onward.
!apt-get -qq install -y git-lfs > /dev/null
import subprocess
from pathlib import Path

ROOT = Path("/content/Project-phase-2")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git", str(ROOT)], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=True)

MPM_DIR = Path("/content/ml_privacy_meter")
if not MPM_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/privacytrustlab/ml_privacy_meter.git", str(MPM_DIR)], check=True)
for c in ("e384af8fd9319b8eeb1303aa82474df1441e3c59", "d32734161a3395211fe5f3cd461932290b1fafbe"):
    subprocess.run(["git", "fetch", "-q", "origin", c], cwd=MPM_DIR, check=True)

%pip install -q "transformers==4.56.2" "peft==0.13.2" "accelerate==1.10.1" "datasets==4.1.1"
print("Done. Restart the session now.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requ

In [2]:
# R-2 — Imports and constants. Nothing inherited from earlier runtimes.
import os, json, time, hashlib, subprocess
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import logsumexp
from scipy.stats import trim_mean
from sklearn.metrics import roc_auc_score, roc_curve
import transformers, peft
from transformers import AutoTokenizer
assert transformers.__version__ == "4.56.2" and peft.__version__ == "0.13.2"

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/Project-phase-2")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
OLD = DRIVE / "cache"                              # read-only here
RMIA_CACHE = DRIVE / "cache" / "rmia"              # all new RMIA outputs
RMIA_CACHE.mkdir(parents=True, exist_ok=True)
MPM_DIR = Path("/content/ml_privacy_meter")
MPM_LIB_COMMIT = "e384af8fd9319b8eeb1303aa82474df1441e3c59"
MPM_RESEARCH_COMMIT = "d32734161a3395211fe5f3cd461932290b1fafbe"

MODEL_NAME = "gpt2_3_lora32_adamw_b8_lr2"
MAX_LENGTH = 512
PHASES = ("train", "val", "final")                 # "train" = the dev target's evaluation texts
PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]
REF_NAMES = [n for p in PAIRS for n in p]
PARTNER = {a: b for a, b in PAIRS} | {b: a for a, b in PAIRS}
FRESH_AUX = {"p4A", "p4B"}
GAMMA_GRID = (1.0, 2.0)                            # library value and paper-config value
PROPORTIONTOCUT = 0.2                              # paper configs
print("Paths OK:", ROOT.exists(), DRIVE.exists(), MPM_DIR.exists())

Mounted at /content/drive
Paths OK: True True True


In [3]:
# R-3 — Texts in original order, labels, tokenizer, and actual training membership of every paired shadow.
def read_json(p):
    with Path(p).open("r", encoding="utf-8") as f:
        return json.load(f)

TEXTS = {ph: read_json(ROOT / "data" / ph / "test.json") for ph in PHASES}
LABELS = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
assert all(len(TEXTS[ph]) == 2000 for ph in PHASES) and LABELS.sum() == 1000
DEV_ORDER_SHA = hashlib.sha256("\x1e".join(TEXTS["train"]).encode()).hexdigest()

ADAPTER_DIR = ROOT / "models" / "train" / MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

norm = lambda t: " ".join(t.split())
def prefix_keys(texts):
    keys = []
    for s in range(0, len(texts), 256):
        for ids in tokenizer(texts[s:s + 256], add_special_tokens=False, truncation=True,
                             max_length=MAX_LENGTH, padding=False)["input_ids"]:
            keys.append(hashlib.sha256(np.asarray(ids, dtype="<i4").tobytes()).hexdigest())
    return keys

EVAL_NORM = {ph: [norm(t) for t in TEXTS[ph]] for ph in PHASES}
EVAL_PREF = {ph: prefix_keys(TEXTS[ph]) for ph in PHASES}

def membership(train_texts):
    ns = {norm(t) for t in train_texts}; ps = set(prefix_keys(list(train_texts)))
    return {ph: np.array([(a in ns) or (b in ps) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])]) for ph in PHASES}

MEM = {n: membership([r["text"] for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")])
       for n in REF_NAMES}
for a, b in PAIRS[1:]:
    for ph in PHASES:
        assert (MEM[a][ph] ^ MEM[b][ph]).all(), (a, b, ph)          # new pairs: exactly one IN
assert int((MEM["inA"]["train"] & MEM["inB"]["train"]).sum()) == 7     # known exception in pair 1
for ph in PHASES:
    assert (sum(MEM[n][ph].astype(int) for n in ("inA", "inB")) >= 1).all()
print("Membership loaded; pair constraints hold.")

Membership loaded; pair constraints hold.


In [4]:
# R-4 — Cached per-token losses (float64 for summation), alignment checks, frozen LR4 dev scores.
def split_npz(path, key="losses"):
    z = np.load(path)
    return [a.astype(np.float64) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

TGT = {"train": split_npz(OLD / "train_base_target_losses.npz", "target_losses"),   # dev target on dev texts
       "val":   split_npz(OLD / "val_ref_train_losses.npz"),                         # dev target on val texts
       "final": split_npz(OLD / "final_train_tgt_losses.npz")}                       # dev target on final texts
REF = {n: {ph: split_npz(OLD / f"{ph}_shadow_{n}_losses.npz") for ph in PHASES} for n in REF_NAMES}

for ph in PHASES:
    exp_len = np.array([len(ids) - 1 for ids in tokenizer(TEXTS[ph], truncation=True, max_length=MAX_LENGTH)["input_ids"]])
    got = np.array([a.size for a in TGT[ph]])
    assert np.array_equal(exp_len, got), f"target cache misaligned on {ph}"
    for n in REF_NAMES:
        assert len(REF[n][ph]) == 2000 and all(r.shape == t.shape for r, t in zip(REF[n][ph], TGT[ph])), (n, ph)
LEN = {ph: np.array([a.size for a in TGT[ph]], dtype=np.float64) for ph in PHASES}
print("All caches aligned with current tokenization and example order.")

def tpr_strict(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr[fpr < 0.01]))

S4 = np.load(OLD / "staged4_dev_scores_4pairs.npz")
LR4_DEV = S4["LR (all OUT, κ=8 fixed)"].astype(np.float64)
B5_SAVED = S4["B5"].astype(np.float64)
settings = read_json(DRIVE / "frozen_LR4_k8" / "settings.json")
assert LR4_DEV.shape == (2000,) and np.isclose(tpr_strict(LABELS, LR4_DEV), settings["dev"]["LR"])

# Order check for LR4: rebuild B5 from per-text caches in dev order and match the B5 saved with LR4.
fixed = [split_npz(OLD / "train_shadow_1001_losses.npz", "shadow_losses"), split_npz(OLD / "train_shadow_1002_losses.npz"),
         split_npz(OLD / "train_ref_val_losses.npz"), split_npz(OLD / "train_ref_final_losses.npz")]
b5 = np.empty(2000)
for i, t in enumerate(TGT["train"]):
    outs = [r[i] for r in fixed] + [REF[n]["train"][i] for n in ("inA", "inB") if not MEM[n]["train"][i]]
    b5[i] = (np.stack(outs).astype(np.float32).mean(0) - t.astype(np.float32)).mean() * np.sqrt(t.size)
assert np.allclose(b5, B5_SAVED, atol=1e-4), np.abs(b5 - B5_SAVED).max()
print(f"LR4 order verified. LR4 dev: AUC={roc_auc_score(LABELS, LR4_DEV):.4f} | strict TPR={tpr_strict(LABELS, LR4_DEV):.4f}")

All caches aligned with current tokenization and example order.
LR4 order verified. LR4 dev: AUC=0.9744 | strict TPR=0.7120


In [5]:
# R-5 — Both statistics from the same caches; population Z under two selection rules.
STATS = {
    "mean": "−mean(token losses) = log of the geometric-mean token probability (ml_privacy_meter LM signal; primary)",
    "sum":  "−sum(token losses) = conditional log-likelihood of tokens 2..T given token 1 "
            "(sequence-likelihood adaptation; secondary; not the full text probability)",
}
def stat_vals(arrs, stat):
    return np.array([(-a.mean() if stat == "mean" else -a.sum()) for a in arrs])

X_T = {s: {ph: stat_vals(TGT[ph], s) for ph in PHASES} for s in STATS}
X_R = {s: {ph: np.stack([stat_vals(REF[n][ph], s) for n in REF_NAMES]) for ph in PHASES} for s in STATS}
IN_R = {ph: np.stack([MEM[n][ph] for n in REF_NAMES]) for ph in PHASES}
assert all(np.isfinite(X_T[s][ph]).all() and np.isfinite(X_R[s][ph]).all() for s in STATS for ph in PHASES)

n_in_dev = IN_R["train"].sum(0)
print("Dev texts by # IN among the 8 references:", {int(k): int((n_in_dev == k).sum()) for k in np.unique(n_in_dev)})

train_ft_texts = [r if isinstance(r, str) else r["text"] for r in read_json(ROOT / "data" / "train" / "train_finetune.json")]
TFT_NORM, TFT_PREF = {norm(t) for t in train_ft_texts}, set(prefix_keys(train_ft_texts))
DEV_NORM, DEV_PREF = set(EVAL_NORM["train"]), set(EVAL_PREF["train"])
dup_dev = {ph: np.array([(a in DEV_NORM) or (b in DEV_PREF) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])])
           for ph in ("val", "final")}
in_tft = {ph: np.array([(a in TFT_NORM) or (b in TFT_PREF) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])])
          for ph in ("val", "final")}

Z_RULES = {
    "Z_devknown": {ph: ~dup_dev[ph] & ~in_tft[ph] for ph in ("val", "final")},   # uses dev target's training file
    "Z_transfer": {ph: ~dup_dev[ph] for ph in ("val", "final")},                 # available for any target
}
for name, zm in Z_RULES.items():
    leaked = sum(int((zm[ph] & in_tft[ph]).sum()) for ph in ("val", "final"))
    print(f"{name}: |Z| = {sum(int(m.sum()) for m in zm.values())} | known dev-target members inside Z = {leaked}")

Dev texts by # IN among the 8 references: {4: 1993, 5: 7}
Z_devknown: |Z| = 3922 | known dev-target members inside Z = 0
Z_transfer: |Z| = 3983 | known dev-target members inside Z = 61


In [6]:
# R-6 — a(x), population-fraction score, and LR; each verified before use.
def log_trim_mean_exp(L, prop=PROPORTIONTOCUT):
    """log of scipy.stats.trim_mean(exp(L), prop, axis=0); sorting log-values is equivalent (monotone)."""
    K = L.shape[0]; c = int(prop * K)
    S = np.sort(L, axis=0)[c:K - c]
    return logsumexp(S, axis=0) - np.log(S.shape[0])

def a_score(lt, lref):
    """Continuous normalized log-ratio a(x) = log p_θ(x) − log p̂(x). Independent of Z and γ."""
    return lt - log_trim_mean_exp(lref)

def rmia_fraction(a_x, a_z, gamma):
    """Official RMIA score: fraction of z with LR(x,z) > γ  ⇔  a_z < a(x) − log γ  (= F̂_Z(a(x) − log γ))."""
    az = np.sort(a_z)
    return np.searchsorted(az, a_x - np.log(gamma), side="left") / az.size

# Check 1: probability-space mirror of research/2024_rmia/main.py (online branch) on synthetic data.
rng_v = np.random.default_rng(0)
K, NX, NZ = 8, 300, 400
pr_x, pt_x = rng_v.uniform(0.05, 0.95, (K, NX)), rng_v.uniform(0.05, 0.95, NX)
pr_z, pt_z = rng_v.uniform(0.05, 0.95, (K, NZ)), rng_v.uniform(0.05, 0.95, NZ)
for g in GAMMA_GRID:
    mx, mz = trim_mean(pr_x, PROPORTIONTOCUT, axis=0), trim_mean(pr_z, PROPORTIONTOCUT, axis=0)
    ref_scores = ((np.outer(pt_x / mx, 1 / (pt_z / mz)) > g) * 1.0).mean(1)
    ours = rmia_fraction(a_score(np.log(pt_x), np.log(pr_x)), a_score(np.log(pt_z), np.log(pr_z)), g)
    assert np.max(np.abs(ref_scores - ours)) <= 1.0 / NZ + 1e-12, g
    ax = a_score(np.log(pt_x), np.log(pr_x)); o = np.argsort(ax)
    assert np.all(np.diff(ours[o]) >= 0), "fraction score must be nondecreasing in a(x)"
print("RMIA scoring matches the research-code formula; fraction score is a nondecreasing map of a(x).")

# LR (Gaussian, κ shrinkage) — same arithmetic as the frozen LR4, with masked missing references.
def lr_scores(mt, L, P, Pin, kappa=8):
    sq = np.sqrt(L); U, ut = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    v_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0) / np.maximum(df_in, 1)
    v_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0) / np.maximum(df_out, 1)
    pool_out = (v_out * df_out).sum() / df_out.sum()
    pool_in = (v_in * df_in).sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = (v_in * df_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = (v_out * df_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    return (-(ut - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(ut - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))

DEV_MT = -X_T["mean"]["train"]                                         # target mean token loss
DEV_L = LEN["train"]
P8 = -X_R["mean"]["train"]                                             # 8 paired shadows' mean token loss
PIN8 = IN_R["train"]
FIX4 = np.stack([np.array([a.mean() for a in f]) for f in fixed])     # shadow1001, shadow1002, val tgt, final tgt (R-4)

# Check 2: reproduce frozen LR4 exactly (12 references), then compute the equal-reference LR (8 paired only).
lr4_rebuilt = lr_scores(DEV_MT, DEV_L, np.vstack([P8, FIX4]), np.vstack([PIN8, np.zeros(FIX4.shape, bool)]))
assert np.allclose(lr4_rebuilt, LR4_DEV, rtol=0, atol=1e-6), np.abs(lr4_rebuilt - LR4_DEV).max()
assert np.isclose(tpr_strict(LABELS, lr4_rebuilt), tpr_strict(LABELS, LR4_DEV))
LR8_DEV = lr_scores(DEV_MT, DEV_L, P8, PIN8)
print("LR implementation reproduces frozen LR4; equal-reference LR (8 paired shadows) computed.")

RMIA scoring matches the research-code formula; fraction score is a nondecreasing map of a(x).
LR implementation reproduces frozen LR4; equal-reference LR (8 paired shadows) computed.


In [7]:
# R-7 — Dev evaluation: both statistics, a(x) diagnostic, fraction score (γ=1,2; two Z rules), LR-8, frozen LR4.
t0 = time.perf_counter()
SCORES, META = {}, {}
for s in STATS:
    a_x = a_score(X_T[s]["train"], X_R[s]["train"])
    SCORES[f"{s} | a(x) continuous [diagnostic]"] = a_x
    META[f"{s} | a(x) continuous [diagnostic]"] = (s, "a(x)", "-", "-")
    for zname, zm in Z_RULES.items():
        a_z = np.concatenate([a_score(X_T[s][ph][zm[ph]], X_R[s][ph][:, zm[ph]]) for ph in ("val", "final")])
        for g in GAMMA_GRID:
            key = f"{s} | RMIA fraction | γ={g:g} | {zname}"
            SCORES[key] = rmia_fraction(a_x, a_z, g)
            META[key] = (s, "fraction", g, zname)
score_seconds = time.perf_counter() - t0
SCORES["LR κ=8 | same 8 paired refs"] = LR8_DEV;      META["LR κ=8 | same 8 paired refs"] = ("mean", "LR", "-", "-")
SCORES["LR4 frozen | 12 refs"] = LR4_DEV;             META["LR4 frozen | 12 refs"] = ("mean", "LR", "-", "-")
for k, v in SCORES.items():
    assert v.shape == (2000,) and np.isfinite(v).all(), k

# Dominance of the trimmed log-mean-exp by the top reference, per statistic.
for s in STATS:
    S = np.sort(X_R[s]["train"], axis=0)[int(PROPORTIONTOCUT * 8):8 - int(PROPORTIONTOCUT * 8)]
    print(f"[{s}] median gap between top two retained references: {np.median(S[-1] - S[-2]):.3f} nats")

def strict_fast(sc, y):
    neg, pos = sc[y == 0], sc[y == 1]
    return np.mean(pos > np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1])
for k, v in SCORES.items():
    assert np.isclose(strict_fast(v, LABELS), tpr_strict(LABELS, v)), k

rng = np.random.default_rng(20261017)
pos_i, neg_i = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
boot = [np.concatenate([rng.choice(pos_i, pos_i.size), rng.choice(neg_i, neg_i.size)]) for _ in range(2000)]
BT = {k: np.array([strict_fast(v[ix], LABELS[ix]) for ix in boot]) for k, v in SCORES.items()}

rows = []
for k, v in SCORES.items():
    stat, kind, g, zname = META[k]
    r = {"method": k, "dev AUC": roc_auc_score(LABELS, v), "strict TPR@FPR<0.01": tpr_strict(LABELS, v),
         "unique": int(np.unique(v).size),
         "frac=0": float(np.mean(v == 0)) if kind == "fraction" else np.nan,
         "frac=1": float(np.mean(v == 1)) if kind == "fraction" else np.nan}
    for ref in ("LR κ=8 | same 8 paired refs", "LR4 frozen | 12 refs"):
        lab = "LR8" if ref.startswith("LR κ") else "LR4"
        d = BT[k] - BT[ref]; lo, hi = np.percentile(d, [2.5, 97.5])
        r[f"Δ vs {lab}"] = tpr_strict(LABELS, v) - tpr_strict(LABELS, SCORES[ref])
        r[f"95% CI vs {lab}"] = f"[{lo:+.3f}, {hi:+.3f}]"
    rows.append(r)
res = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

n_z = {zn: sum(int(m.sum()) for m in zm.values()) for zn, zm in Z_RULES.items()}
print(f"\nCompute: no training, no new inference. RMIA scoring time (both statistics, both Z rules, both γ): "
      f"{score_seconds:.2f} s. Cached forward passes used: target + 8 paired shadows on 2000 dev texts and up to "
      f"{max(n_z.values())} population texts. LR-8 uses target + 8 shadows on dev only; LR4 adds 4 OUT-only references "
      f"(shadows 1001/1002, val and final targets). Reference training already done: ~25 min per shadow on a T4.")

safe = {k.replace(" ", "_").replace("|", "").replace("=", "").replace("κ", "k").replace("γ", "g"): v for k, v in SCORES.items()}
np.savez(RMIA_CACHE / "rmia_dev_scores.npz", labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA), **safe)
res.to_csv(RMIA_CACHE / "rmia_dev_vs_LR.csv", index=False)
json.dump({
    "sources": {"library": f"privacytrustlab/ml_privacy_meter@{MPM_LIB_COMMIT}",
                "paper_code": f"ml_privacy_meter@{MPM_RESEARCH_COMMIT} research/2024_rmia/main.py (online branch)"},
    "configuration": {"variant": "online", "references": REF_NAMES, "p_hat": "trimmed mean, proportiontocut=0.2, "
                      "probability space via log-mean-exp", "gamma": "fixed settings 1 and 2 (no selection)",
                      "statistics": STATS, "score": "fraction of Z with LR(x,z) > γ (official); a(x) reported as a "
                      "diagnostic variant", "population_rules": {
                          "Z_devknown": "val+final texts minus duplicates of dev texts minus texts in the dev target's "
                                        "released training file — NOT transferable to val/final targets",
                          "Z_transfer": "val+final texts minus duplicates of the scored texts only — transferable"},
                      "n_z": n_z},
    "equal_reference_comparison": "LR κ=8 on the same 8 paired shadows; frozen LR4 additionally uses 4 OUT-only references",
    "data_use": "Uses the dev evaluation texts with their public labels (evaluation only); the val and final evaluation "
                "TEXTS as population candidates and reference inputs; the dev target's released training file for "
                "Z_devknown filtering. No hidden validation or final membership labels; no validation submission.",
    "notes": ["The fraction score is a nondecreasing function of a(x) because every x uses the same Z; it cannot improve "
              "a(x)'s ordering, only add ties/saturation.",
              "−sum(losses) is the conditional log-likelihood of tokens 2..T given token 1, not the full text probability.",
              "No augmentation (paper uses image augmentations)."],
    "new_training": 0, "new_inference": 0,
}, open(RMIA_CACHE / "rmia_notes.json", "w"), indent=2)
print("Saved to", RMIA_CACHE)

[mean] median gap between top two retained references: 0.009 nats
[sum] median gap between top two retained references: 1.157 nats
                                 method  dev AUC  strict TPR@FPR<0.01  unique  frac=0  frac=1  Δ vs LR8    95% CI vs LR8  Δ vs LR4    95% CI vs LR4
    mean | a(x) continuous [diagnostic]   0.9655               0.4730    2000     NaN     NaN   -0.1930 [-0.377, -0.031]   -0.2390 [-0.415, -0.070]
mean | RMIA fraction | γ=1 | Z_devknown   0.9655               0.4700     941  0.0000  0.0140   -0.1960 [-0.382, -0.038]   -0.2420 [-0.420, -0.074]
mean | RMIA fraction | γ=2 | Z_devknown   0.5000               0.0000       1  1.0000  0.0000   -0.6660 [-0.771, -0.580]   -0.7120 [-0.790, -0.597]
mean | RMIA fraction | γ=1 | Z_transfer   0.9655               0.4700     989  0.0000  0.0045   -0.1960 [-0.382, -0.037]   -0.2420 [-0.420, -0.074]
mean | RMIA fraction | γ=2 | Z_transfer   0.5000               0.0000       1  1.0000  0.0000   -0.6660 [-0.771, -0.580]   -0.712

In [8]:
# LRV — LR variance variants with LR4's exact references, masks, √T-scaled mean losses and κ=8.
# A: unequal-variance LR (= frozen LR4). B: shared-variance LR (distinct variant). C: OUT-only standardized gap.
LRV_CACHE = DRIVE / "cache" / "lr_variants"; LRV_CACHE.mkdir(parents=True, exist_ok=True)
KAPPA = 8

def lr_family(mt, L, P, Pin, kappa=KAPPA):
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin          # missing references count as neither
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0)
    rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)

    # A — unequal variances, each shrunk to its own pooled value, × (1+1/n): identical to LR4.
    pool_out = rss_out.sum() / df_out.sum()
    pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    shr_in = (rss_in + kappa * pool_in) / (df_in + kappa)
    shr_out = (rss_out + kappa * pool_out) / (df_out + kappa)
    s_in, s_out = shr_in * (1 + 1 / n_in), shr_out * (1 + 1 / n_out)
    A = (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))

    # B — one shared within-group variance per text: (RSS_in + RSS_out)/(df_in + df_out), shrunk (κ) to the
    #     global pooled within-group variance. No 1+1/n factors (they would reintroduce unequal variances).
    df_sh = df_in + df_out
    assert df_sh.sum() > 0
    pool_sh = (rss_in + rss_out).sum() / df_sh.sum()
    v_sh = (rss_in + rss_out + kappa * pool_sh) / (df_sh + kappa)
    B = ((u - mu_out) ** 2 - (u - mu_in) ** 2) / (2 * v_sh)

    # C — OUT-only standardized gap with the existing shrunk OUT predictive variance.
    C = (mu_out - u) / np.sqrt(s_out)

    for x in (A, B, C):
        assert np.isfinite(x).all()
    diag = {"pool_in": pool_in, "pool_out": pool_out, "pool_shared": pool_sh,
            "texts_mu_in_ge_mu_out": int((mu_in >= mu_out).sum())}   # B's slope sign flips for these
    return {"A": A, "B": B, "C": C}, diag

NAMES = {"A": "A  unequal-variance LR (LR4)", "B": "B  shared-variance LR", "C": "C  OUT-only standardized gap"}

# ---------- Dev: exact LR4 reference set (8 paired + shadows 1001/1002 + val/final targets) ----------
P_full = np.vstack([P8, FIX4]); PIN_full = np.vstack([PIN8, np.zeros(FIX4.shape, bool)])
dev, ddiag = lr_family(DEV_MT, DEV_L, P_full, PIN_full)
assert np.allclose(dev["A"], LR4_DEV, rtol=0, atol=1e-6), np.abs(dev["A"] - LR4_DEV).max()
assert np.isclose(tpr_strict(LABELS, dev["A"]), tpr_strict(LABELS, LR4_DEV))
print("Variant A reproduces frozen LR4. Pooled variances:",
      {k: round(v, 4) if isinstance(v, float) else v for k, v in ddiag.items()})

bt = {k: np.array([strict_fast(v[ix], LABELS[ix]) for ix in boot]) for k, v in dev.items()}
rows = []
for k, v in dev.items():
    d = bt[k] - bt["A"]; lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"method": NAMES[k], "dev AUC": roc_auc_score(LABELS, v), "strict TPR@FPR<0.01": tpr_strict(LABELS, v),
                 "Δ TPR vs LR4": tpr_strict(LABELS, v) - tpr_strict(LABELS, dev["A"]),
                 "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "bootstrap fraction Δ>0": float(np.mean(d > 0))})
dev_tab = pd.DataFrame(rows)
print("\nDev (LR4's exact 12 references, κ=8):")
print(dev_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---------- Matched mock comparison (STAGED-5 setup): refs = inA,inB,p2A,p2B + shadows 1001/1002 ----------
R0 = ["inA", "inB", "p2A", "p2B"]
i0 = [REF_NAMES.index(n) for n in R0]
P0 = np.vstack([P8[i0], FIX4[:2]]); PIN0 = np.vstack([PIN8[i0], np.zeros((2, 2000), bool)])
targets = {"p3A": ("pooled", P8[REF_NAMES.index("p3A")], MEM["p3A"]["train"].astype(int)),
           "p3B": ("pooled", P8[REF_NAMES.index("p3B")], MEM["p3B"]["train"].astype(int)),
           "p4A": ("fresh",  P8[REF_NAMES.index("p4A")], MEM["p4A"]["train"].astype(int)),
           "p4B": ("fresh",  P8[REF_NAMES.index("p4B")], MEM["p4B"]["train"].astype(int)),
           "dev target (real)": ("real", DEV_MT, LABELS)}
STAGED5_LR = {"p3A": 0.8483, "p3B": 0.8084, "p4A": 0.7450, "p4B": 0.6586, "dev target (real)": 0.6590}

mrows, mock_scores = [], {}
for t, (src, mt, y) in targets.items():
    sc, _ = lr_family(mt, DEV_L, P0, PIN0)
    assert abs(tpr_strict(y, sc["A"]) - STAGED5_LR[t]) < 6e-4, (t, tpr_strict(y, sc["A"]))   # reproduces STAGED-5
    for k, v in sc.items():
        mock_scores[f"{t}|{k}"] = v
        mrows.append({"target": t, "aux": src, "method": k, "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v),
                      "Δ TPR vs A": tpr_strict(y, v) - tpr_strict(y, sc["A"])})
mock_tab = pd.DataFrame(mrows)
print("\nMatched mock comparison (STAGED-5 references; A reproduces STAGED-5 LR):")
print(mock_tab.pivot(index=["aux", "target"], columns="method", values="TPR").to_string(float_format=lambda x: f"{x:.4f}"))
print("\nMean Δ TPR vs A by group:")
print(mock_tab[mock_tab.method != "A"].groupby(["aux", "method"])["Δ TPR vs A"].mean().unstack().to_string(float_format=lambda x: f"{x:+.4f}"))

# ---------- Save (separate files; LR4 artifacts untouched) ----------
np.savez(LRV_CACHE / "lr_variants_dev_scores.npz", A_lr4=dev["A"], B_shared=dev["B"], C_out_only=dev["C"],
         labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA))
np.savez(LRV_CACHE / "lr_variants_matched_mock_scores.npz", **{k.replace(" ", "_").replace("(", "").replace(")", ""): v
                                                               for k, v in mock_scores.items()})
dev_tab.to_csv(LRV_CACHE / "lr_variants_dev.csv", index=False)
mock_tab.to_csv(LRV_CACHE / "lr_variants_matched_mock.csv", index=False)
json.dump({"references_dev": REF_NAMES + ["shadow1001", "shadow1002", "val target", "final target"],
           "references_mock": R0 + ["shadow1001", "shadow1002"], "kappa": KAPPA,
           "statistic": "u = sqrt(T) × mean token loss; per-text means over eligible references by actual membership",
           "A": "unequal-variance Gaussian LR; each variance shrunk to its pooled value (κ) × (1+1/n) — frozen LR4",
           "B": "shared-variance LR: v(x) = (RSS_in+RSS_out+κ·pool_shared)/(df_in+df_out+κ); "
                "s = ((u−μ_OUT)² − (u−μ_IN)²)/(2v); no 1+1/n factors; distinct scoring variant",
           "C": "OUT-only: (μ_OUT − u)/sqrt(shrunk OUT var × (1+1/n_OUT)); not previously tested "
                "(closest earlier: Cell 31 blended-sd per-text z with 3–4 OUT refs, no κ, no 1+1/n)",
           "dev_diag": {k: (float(v) if not isinstance(v, int) else v) for k, v in ddiag.items()},
           "no_sweep": True, "validation_submission": False}, open(LRV_CACHE / "lr_variants_notes.json", "w"), indent=2)
print("\nSaved to", LRV_CACHE)

Variant A reproduces frozen LR4. Pooled variances: {'pool_in': np.float64(0.0589), 'pool_out': np.float64(0.0455), 'pool_shared': np.float64(0.0495), 'texts_mu_in_ge_mu_out': 0}

Dev (LR4's exact 12 references, κ=8):
                      method  dev AUC  strict TPR@FPR<0.01  Δ TPR vs LR4           95% CI  bootstrap fraction Δ>0
A  unequal-variance LR (LR4)   0.9744               0.7120        0.0000 [+0.000, +0.000]                  0.0000
       B  shared-variance LR   0.9750               0.7170        0.0050 [-0.036, +0.082]                  0.5425
C  OUT-only standardized gap   0.9721               0.6710       -0.0410 [-0.081, +0.031]                  0.1320

Matched mock comparison (STAGED-5 references; A reproduces STAGED-5 LR):
method                        A      B      C
aux    target                                
fresh  p4A               0.7450 0.7390 0.6902
       p4B               0.6586 0.6446 0.6737
pooled p3A               0.8483 0.8401 0.7424
       p3B             

In [9]:
# TLR — Token-level Gaussian LR with LR4's references, membership masks and κ=8. Dev + matched mocks. LR4 untouched.
TLR_CACHE = DRIVE / "cache" / "lr_token"; TLR_CACHE.mkdir(parents=True, exist_ok=True)
KAPPA = 8
VAR_FLOOR = 1e-6   # nats²; applied to each shrunk token variance before the (1+1/n) factor; guards division/log only

def token_lr(tgt, rows, ins, kappa=KAPPA, floor=VAR_FLOOR):
    """tgt: per-text token losses of the target. rows: list of reference models, each a per-text list of token-loss
    arrays. ins: per reference, bool (N,) actual membership. Returns mean-token-LR, mean-token-LR×√T, diagnostics."""
    N, K = len(tgt), len(rows)
    stats, pin_terms, pout_terms = [], [], []
    for i in range(N):
        X = np.stack([r[i] for r in rows])                                   # (K, T_i)
        assert X.shape[1] == tgt[i].size
        fin = np.isfinite(X).all(1)                                          # missing reference → excluded for text
        is_in = np.array([ins[k][i] for k in range(K)], dtype=bool)
        m_in, m_out = is_in & fin, ~is_in & fin
        n_in, n_out = int(m_in.sum()), int(m_out.sum())
        assert n_in > 0 and n_out > 0, i
        Xi, Xo = X[m_in], X[m_out]
        mu_i, mu_o = Xi.mean(0), Xo.mean(0)
        rss_i, rss_o = ((Xi - mu_i) ** 2).sum(0), ((Xo - mu_o) ** 2).sum(0)
        stats.append((mu_i, mu_o, rss_i, rss_o, n_in, n_out))
        if n_in > 1:  pin_terms.append(rss_i.mean() / (n_in - 1))          # text's mean within-group token variance
        if n_out > 1: pout_terms.append(rss_o.mean() / (n_out - 1))
    pool_out = float(np.mean(pout_terms))                                    # texts weighted equally
    pool_in = float(np.mean(pin_terms)) if pin_terms else pool_out
    S_mean, S_sqrt = np.empty(N), np.empty(N)
    floor_hits = 0
    for i, (mu_i, mu_o, rss_i, rss_o, n_in, n_out) in enumerate(stats):
        v_i = (rss_i + kappa * pool_in) / ((n_in - 1) + kappa)
        v_o = (rss_o + kappa * pool_out) / ((n_out - 1) + kappa)
        floor_hits += int((v_i < floor).sum() + (v_o < floor).sum())
        s_i = np.maximum(v_i, floor) * (1 + 1 / n_in)
        s_o = np.maximum(v_o, floor) * (1 + 1 / n_out)
        x = tgt[i]
        ell = (-(x - mu_i) ** 2 / (2 * s_i) - 0.5 * np.log(s_i)) - (-(x - mu_o) ** 2 / (2 * s_o) - 0.5 * np.log(s_o))
        S_mean[i] = ell.mean()
        S_sqrt[i] = ell.mean() * np.sqrt(x.size)
    assert np.isfinite(S_mean).all() and np.isfinite(S_sqrt).all()
    n_in_all = np.array([s[4] for s in stats]); n_out_all = np.array([s[5] for s in stats])
    return S_mean, S_sqrt, {"pool_in": pool_in, "pool_out": pool_out, "floor_hits": floor_hits,
                            "n_in": n_in_all, "n_out": n_out_all}

ZEROS = np.zeros(2000, dtype=bool)

# ---------- Full dev: LR4's exact references (8 paired by membership + 4 OUT-only) ----------
rows_dev = [REF[n]["train"] for n in REF_NAMES] + fixed
ins_dev = [MEM[n]["train"] for n in REF_NAMES] + [ZEROS] * len(fixed)
t0 = time.perf_counter()
TL_mean, TL_sqrt, tdiag = token_lr(TGT["train"], rows_dev, ins_dev)
secs = time.perf_counter() - t0
assert np.array_equal(tdiag["n_in"], PIN_full.sum(0)) and np.array_equal(tdiag["n_out"], (~PIN_full).sum(0))
print(f"Token pools (texts weighted equally): IN {tdiag['pool_in']:.4f}, OUT {tdiag['pool_out']:.4f} nats² | "
      f"variance-floor hits: {tdiag['floor_hits']} | scoring time {secs:.1f}s")

cands = {"LR4 (frozen, text-level)": LR4_DEV, "token LR, mean": TL_mean, "token LR, mean × √T": TL_sqrt}
btk = {k: np.array([strict_fast(v[ix], LABELS[ix]) for ix in boot]) for k, v in cands.items()}
rows = []
for k, v in cands.items():
    d = btk[k] - btk["LR4 (frozen, text-level)"]; lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"method": k, "dev AUC": roc_auc_score(LABELS, v), "strict TPR@FPR<0.01": tpr_strict(LABELS, v),
                 "Δ TPR vs LR4": tpr_strict(LABELS, v) - tpr_strict(LABELS, LR4_DEV),
                 "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "bootstrap fraction Δ>0": float(np.mean(d > 0))})
tl_dev = pd.DataFrame(rows)
print("\nFull dev (LR4's 12 references, κ=8):")
print(tl_dev.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---------- Matched mock comparison (STAGED-5 references: inA, inB, p2A, p2B + shadows 1001/1002) ----------
rows0 = [REF[n]["train"] for n in R0] + fixed[:2]
ins0 = [MEM[n]["train"] for n in R0] + [ZEROS, ZEROS]
mtargets = {"p3A": ("pooled", REF["p3A"]["train"], MEM["p3A"]["train"].astype(int)),
            "p3B": ("pooled", REF["p3B"]["train"], MEM["p3B"]["train"].astype(int)),
            "p4A": ("fresh",  REF["p4A"]["train"], MEM["p4A"]["train"].astype(int)),
            "p4B": ("fresh",  REF["p4B"]["train"], MEM["p4B"]["train"].astype(int)),
            "dev target (real)": ("real", TGT["train"], LABELS)}
mrows, msc = [], {}
for t, (src, toks, y) in mtargets.items():
    mt = np.array([a.mean() for a in toks])
    base, _ = lr_family(mt, DEV_L, P0, PIN0)                                  # text-level LR (A), as in STAGED-5
    assert abs(tpr_strict(y, base["A"]) - STAGED5_LR[t]) < 6e-4, t
    s_mean, s_sqrt, _ = token_lr(toks, rows0, ins0)
    for name, v in (("text LR (A)", base["A"]), ("token LR, mean", s_mean), ("token LR, mean × √T", s_sqrt)):
        msc[f"{t}|{name}"] = v
        mrows.append({"target": t, "aux": src, "method": name, "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v),
                      "Δ TPR vs text LR": tpr_strict(y, v) - tpr_strict(y, base["A"])})
tl_mock = pd.DataFrame(mrows)
print("\nMatched mock comparison:")
print(tl_mock.pivot(index=["aux", "target"], columns="method", values="TPR").to_string(float_format=lambda x: f"{x:.4f}"))
print("\nMean Δ TPR vs text-level LR by group:")
print(tl_mock[tl_mock.method != "text LR (A)"].groupby(["aux", "method"])["Δ TPR vs text LR"].mean()
      .unstack().to_string(float_format=lambda x: f"{x:+.4f}"))

# ---------- Save ----------
np.savez(TLR_CACHE / "token_lr_dev_scores.npz", token_lr_mean=TL_mean, token_lr_mean_sqrtT=TL_sqrt,
         lr4=LR4_DEV, labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA))
np.savez(TLR_CACHE / "token_lr_matched_mock_scores.npz",
         **{k.replace(" ", "_").replace("(", "").replace(")", "").replace("×", "x").replace("√", "sqrt"): v
            for k, v in msc.items()})
tl_dev.to_csv(TLR_CACHE / "token_lr_dev.csv", index=False)
tl_mock.to_csv(TLR_CACHE / "token_lr_matched_mock.csv", index=False)
json.dump({
    "references_dev": REF_NAMES + ["shadow1001", "shadow1002", "val target", "final target"],
    "references_mock": R0 + ["shadow1001", "shadow1002"], "kappa": KAPPA, "variance_floor": VAR_FLOOR,
    "token_model": "per valid token: IN/OUT mean and model-to-model variance over eligible references (actual "
                   "membership, finite masks); variance shrunk toward global IN/OUT pools with κ; × (1+1/n); "
                   "ℓ_t = log N(x_t; μ_IN, s_IN) − log N(x_t; μ_OUT, s_OUT), positive = member",
    "pools": "from reference losses only, per evaluation setting; each text's mean within-group token variance, "
             "averaged with equal text weights",
    "scores": {"token LR, mean": "mean_t ℓ_t", "token LR, mean × √T": "mean_t ℓ_t × sqrt(T)"},
    "interpretation": "Aggregates of dependent token terms; NOT a calibrated text-level likelihood ratio. "
                      "Probabilities are never multiplied.",
    "selection_note": "Predefined, unswept variants; still part of an adaptive sequence of comparisons on the same "
                      "dev labels.",
    "dev_diag": {"pool_in": tdiag["pool_in"], "pool_out": tdiag["pool_out"], "floor_hits": tdiag["floor_hits"]},
    "validation_submission": False,
}, open(TLR_CACHE / "token_lr_notes.json", "w"), indent=2)
print("\nSaved to", TLR_CACHE)

Token pools (texts weighted equally): IN 0.0291, OUT 0.0327 nats² | variance-floor hits: 0 | scoring time 0.3s

Full dev (LR4's 12 references, κ=8):
                  method  dev AUC  strict TPR@FPR<0.01  Δ TPR vs LR4           95% CI  bootstrap fraction Δ>0
LR4 (frozen, text-level)   0.9744               0.7120        0.0000 [+0.000, +0.000]                  0.0000
          token LR, mean   0.8732               0.1910       -0.5210 [-0.613, -0.373]                  0.0000
     token LR, mean × √T   0.8623               0.2360       -0.4760 [-0.573, -0.351]                  0.0000

Matched mock comparison:
method                    text LR (A)  token LR, mean  token LR, mean × √T
aux    target                                                             
fresh  p4A                     0.7450          0.1086               0.1773
       p4B                     0.6586          0.1094               0.1737
pooled p3A                     0.8483          0.1629               0.1843
       p3B